# 23 — Feature-parity worklist freeze (CPU, fail-closed)

Notebook này nối notebook 22 với các producer feature còn thiếu. Nó không chạy
model, không đọc held-out, không sửa `keyframes-btc` và chưa chấm B0–B4. Nó xác
minh selection freeze của notebook 12, hai runner JPEG + SigLIP của notebook 13,
và COMPLETE của notebook 22; sau đó khóa đúng 17.399 frame thành ba workload:

1. `ocr_classical`: EasyOCR + PaddleOCR + VietOCR.
2. `qwen25`: caption + OCR Qwen2.5-VL.
3. `bge_m3`: chỉ chạy sau `qwen25.COMPLETE`.

Notebook fail-closed trên manifest, marker payload, SHA-256 đã được notebook 13
verify trước khi ghi marker, và sự tồn tại hiện tại của `images`,
`image_manifest`, `siglip`, `report`. Nó không tải lại 14,7 GiB chỉ để băm lần
nữa. Thành công chỉ có nghĩa
`READY_FOR_FEATURE_PARITY_PRODUCERS`, chưa phải full parity hay official uplift.

Chạy bằng **Accelerator None**, bật Internet và ba secrets GitHub/Modal. Giữ
nguyên ô 0 rồi Save Version / Run All một lần. Log cuối hợp lệ là
`FEATURE_PARITY_WORKLIST_FROZEN`, `frames=17399`, `runners=2`.

## 0 — Khóa lineage

In [ ]:
from pathlib import Path

RUN_SCOPE = "FREEZE"
CONFIRM_FREEZE = "FEATURE-PARITY-17399"
RUNNERS = ("quang", "trong")
EXPECTED_SELECTION_FREEZE_SHA256 = "75296b93ab4428ae721776d90167cdf3bca6b262137fd93610d04082def5a630"
EXPECTED_CORPUS_LOCK_HASH = "2e0887652464e0cc7942bedb94177cfe2dff5f7baf77081902461cf3c885d75f"
EXPECTED_NB22_COMPLETE_SHA256 = "8731576f70aa01ddcfe59314699143613b56fcaac4f64788641758611e8606e4"
EXPECTED_FEATURE_FRAMES = 17399
EXPECTED_SIGLIP_MODEL_SNAPSHOT_HASH = "605b737a0cb09f9f76ee898fc8ec47117e38c2d8a64b28d3e87977135c055579"
REPO_URL = "https://github.com/AIVIETNAM-AIO-Triet-Descartes/multimodal-fusion-engine.git"
REPO_REF = "feature/quang/storage"
REPO = Path("/kaggle/working/multimodal-fusion-engine")
WORK = Path("/kaggle/working/btc-feature-parity-freeze")
INPUT, OUTPUT = WORK / "input", WORK / "output"
for path in (WORK, INPUT, OUTPUT):
    path.mkdir(parents=True, exist_ok=True)
if RUN_SCOPE != "FREEZE" or CONFIRM_FREEZE != "FEATURE-PARITY-17399":
    raise RuntimeError("Feature-parity freeze lock is closed")

## 1 — Repo và research volume

In [ ]:
import json
import os
import subprocess
import sys
from collections import defaultdict
from datetime import datetime, timezone

def run(command: list[str], *, cwd: Path | None = None, env: dict[str, str] | None = None) -> str:
    result = subprocess.run(command, cwd=cwd, env=env, check=False, text=True, capture_output=True)
    if result.returncode:
        raise RuntimeError(f"Command failed: {' '.join(command[:3])}\n{result.stderr[-1500:]}")
    return result.stdout.strip()

def refresh_repo() -> None:
    from kaggle_secrets import UserSecretsClient
    token = UserSecretsClient().get_secret("GITHUB_TOKEN")
    askpass = WORK / "git-askpass.sh"
    askpass.write_text('#!/bin/sh\ncase "$1" in *Username*) printf x-access-token ;; *) printf %s "$GITHUB_TOKEN" ;; esac\n', encoding="utf-8")
    askpass.chmod(0o700)
    env = os.environ.copy()
    env.update({"GITHUB_TOKEN": token, "GIT_ASKPASS": str(askpass), "GIT_TERMINAL_PROMPT": "0"})
    try:
        if (REPO / ".git").is_dir():
            if run(["git", "status", "--porcelain"], cwd=REPO):
                raise RuntimeError("Checkout Kaggle có local changes; Restart Session rồi Run All")
            run(["git", "fetch", "--depth", "1", "origin", REPO_REF], cwd=REPO, env=env)
            run(["git", "checkout", "--detach", "FETCH_HEAD"], cwd=REPO)
        else:
            run(["git", "clone", "--depth", "1", "--branch", REPO_REF, REPO_URL, str(REPO)], env=env)
    finally:
        askpass.unlink(missing_ok=True)

refresh_repo()
run([sys.executable, "-m", "pip", "install", "-q", "-r", str(REPO / "notebooks/requirements-kaggle-nomodel.txt")])
sys.path.insert(0, str(REPO / "src"))
import yaml
from mfe.offline.keyframe_augmentation.common import atomic_write_json, atomic_write_jsonl, read_jsonl, sha256_file, sha256_json
from mfe.offline.keyframe_augmentation.delta_siglip import partition_feature_worklist_by_runner
from mfe.offline.source import load_catalogue
from mfe.offline.transport import ModalVolumeTransport

REPO_COMMIT = run(["git", "rev-parse", "HEAD"], cwd=REPO)
config = yaml.safe_load((REPO / "config/keyframe_augmentation_kaggle.yaml").read_text(encoding="utf-8"))
catalogue = load_catalogue(REPO / config["source"]["catalogue"])
research = ModalVolumeTransport(config["output"]["volume"], create_if_missing=False)
root = config["output"]["root"].rstrip("/")
selection_root = f"{root}/gate1-selection-v1"
siglip_root = f"{root}/gate1-features-siglip-v1"
nb22_root = f"{root}/gate1-o1s-exact-aggregate-v1"
output_root = f"{root}/gate1-feature-parity-worklist-v1"
print({"repo_commit": REPO_COMMIT, "accelerator": "NONE", "output_root": output_root})

## 2 — Xác minh notebook 12 và 22

In [ ]:
selection_path, worklist_path, nb22_path = INPUT / "selection-freeze.json", INPUT / "feature-worklist.jsonl", INPUT / "nb22-COMPLETE.json"
research.get(f"{selection_root}/selection-freeze.json", selection_path)
research.get(f"{selection_root}/feature-worklist.jsonl", worklist_path)
research.get(f"{nb22_root}/COMPLETE.json", nb22_path)
if sha256_file(selection_path) != EXPECTED_SELECTION_FREEZE_SHA256:
    raise RuntimeError("Notebook 12 selection freeze SHA drift")
if sha256_file(nb22_path) != EXPECTED_NB22_COMPLETE_SHA256:
    raise RuntimeError("Notebook 22 COMPLETE SHA drift")
selection = json.loads(selection_path.read_text(encoding="utf-8"))
unsigned = dict(selection); claimed = unsigned.pop("freeze_payload_hash", None)
if claimed != sha256_json(unsigned) or selection.get("input_corpus_lock_hash") != EXPECTED_CORPUS_LOCK_HASH:
    raise RuntimeError("Notebook 12 payload/corpus lineage invalid")
if sha256_file(worklist_path) != selection["files"]["feature-worklist.jsonl"]:
    raise RuntimeError("Feature worklist SHA mismatch")
nb22 = json.loads(nb22_path.read_text(encoding="utf-8"))
unsigned = dict(nb22); claimed = unsigned.pop("manifest_payload_hash", None)
if claimed != sha256_json(unsigned) or nb22.get("status") != "COMPLETE":
    raise RuntimeError("Notebook 22 COMPLETE payload invalid")
if nb22.get("heldout_accessed") is not False or nb22.get("btc_mutated") is not False:
    raise RuntimeError("Notebook 22 safety lineage invalid")
WORKLIST = list(read_jsonl(worklist_path))
if len(WORKLIST) != EXPECTED_FEATURE_FRAMES or len({row["keyframe_id"] for row in WORKLIST}) != EXPECTED_FEATURE_FRAMES:
    raise RuntimeError("Feature worklist identity/count drift")
for row in WORKLIST:
    payload = {key: value for key, value in row.items() if key != "work_item_hash"}
    if sha256_json(payload) != row["work_item_hash"]:
        raise RuntimeError(f"Work item payload hash mismatch: {row['keyframe_id']}")
PARTITIONS = partition_feature_worklist_by_runner(WORKLIST, RUNNERS, {key: value.size for key, value in catalogue.items()})
EXPECTED_BY_RUNNER, RUNNER_BY_KEYFRAME = {}, {}
for runner, rows in PARTITIONS.items():
    grouped = defaultdict(list)
    for row in rows:
        grouped[row["video_id"]].append(row)
        RUNNER_BY_KEYFRAME[row["keyframe_id"]] = runner
    EXPECTED_BY_RUNNER[runner] = grouped
print({"selection": "PASS", "nb22": "PASS", "frames": len(WORKLIST), "runner_expected": {runner: {"videos": len(grouped), "frames": sum(map(len, grouped.values()))} for runner, grouped in EXPECTED_BY_RUNNER.items()}})

## 3 — Audit notebook 13: manifest, marker và artifact

In [ ]:
def remote_paths(video_id: str, runner: str) -> dict[str, str]:
    return {"images": f"{siglip_root}/images/{video_id}.tar", "image_manifest": f"{siglip_root}/image-manifests/{video_id}.json", "siglip": f"{siglip_root}/siglip/{video_id}.npz", "report": f"{siglip_root}/reports/{video_id}.json", "marker": f"{siglip_root}/markers/{runner}/{video_id}.json"}

RUNNER_AUDIT, MARKER_HASHES, SOURCE_ARTIFACTS = {}, {}, {}
for runner in RUNNERS:
    local_manifest = INPUT / f"runner-manifest-{runner}.json"
    research.get(f"{siglip_root}/runner-manifests/{runner}.json", local_manifest)
    manifest = json.loads(local_manifest.read_text(encoding="utf-8"))
    unsigned = dict(manifest); manifest_hash = unsigned.pop("manifest_payload_hash", None)
    expected = EXPECTED_BY_RUNNER[runner]
    expected_frames = sum(map(len, expected.values()))
    if manifest_hash != sha256_json(unsigned):
        raise RuntimeError(f"{runner}: runner manifest payload hash mismatch")
    checks = (manifest.get("status") == "COMPLETE", manifest.get("runner") == runner, manifest.get("runner_count") == 2, manifest.get("selection_freeze_sha256") == EXPECTED_SELECTION_FREEZE_SHA256, manifest.get("assigned_videos") == len(expected), manifest.get("assigned_frames") == expected_frames, manifest.get("verified_videos") == len(expected), manifest.get("verified_frames") == expected_frames, not manifest.get("failures_this_session"))
    if not all(checks):
        raise RuntimeError(f"{runner}: runner completeness/assignment mismatch")
    canary = manifest.get("siglip_canary_report", {})
    if not str(canary.get("status", "")).startswith("PASS_NUMERIC_CANARY") or canary.get("model_snapshot_hash") != EXPECTED_SIGLIP_MODEL_SNAPSHOT_HASH or float(canary.get("max_abs_diff", 1)) > float(canary.get("tolerance", 0)):
        raise RuntimeError(f"{runner}: SigLIP numeric canary invalid")
    verified = {row["video_id"]: row for row in manifest.get("verified", [])}
    if set(verified) != set(expected):
        raise RuntimeError(f"{runner}: verified video set drift")
    for ordinal, video_id in enumerate(sorted(expected), 1):
        paths = remote_paths(video_id, runner)
        marker_path = INPUT / "markers" / runner / f"{video_id}.json"
        research.get(paths["marker"], marker_path)
        marker = json.loads(marker_path.read_text(encoding="utf-8"))
        unsigned = dict(marker); marker_hash = unsigned.pop("marker_payload_hash", None)
        rows = expected[video_id]
        marker_checks = (marker_hash == sha256_json(unsigned), marker.get("status") == "COMPLETE", marker.get("runner") == runner, marker.get("video_id") == video_id, marker.get("selection_freeze_sha256") == EXPECTED_SELECTION_FREEZE_SHA256, marker.get("model_snapshot_hash") == EXPECTED_SIGLIP_MODEL_SNAPSHOT_HASH, marker.get("frame_count") == len(rows), marker.get("work_items_hash") == sha256_json([row["work_item_hash"] for row in rows]))
        if not all(marker_checks):
            raise RuntimeError(f"{runner}/{video_id}: marker contract mismatch")
        row = verified[video_id]
        if row.get("marker_payload_hash") != marker_hash or row.get("frame_count") != len(rows) or row.get("siglip_sha256") != marker.get("siglip_sha256") or row.get("images_sha256") != marker.get("images_sha256"):
            raise RuntimeError(f"{runner}/{video_id}: manifest/marker mismatch")
        for artifact in ("images", "image_manifest", "siglip", "report"):
            recorded_hash = marker.get(f"{artifact}_sha256")
            if not isinstance(recorded_hash, str) or len(recorded_hash) != 64:
                raise RuntimeError(f"{runner}/{video_id}: invalid recorded {artifact} SHA")
            if not research.exists(paths[artifact]):
                raise RuntimeError(f"{runner}/{video_id}: remote {artifact} is missing")
        MARKER_HASHES[video_id], SOURCE_ARTIFACTS[video_id] = marker_hash, paths
        if ordinal == 1 or ordinal % 50 == 0 or ordinal == len(expected):
            print({"runner": runner, "audited_videos": ordinal, "expected_videos": len(expected)}, flush=True)
    RUNNER_AUDIT[runner] = {"status": "PASS", "manifest_sha256": sha256_file(local_manifest), "manifest_payload_hash": manifest_hash, "videos": len(expected), "frames": expected_frames, "siglip_canary_report_hash": canary.get("report_hash")}
if set(SOURCE_ARTIFACTS) != {row["video_id"] for row in WORKLIST}:
    raise RuntimeError("Notebook 13 artifact video coverage incomplete")
print({"NOTEBOOK13_AUDIT": "PASS", "runners": RUNNER_AUDIT, "marker_count": len(MARKER_HASHES)})

## 4 — Freeze ba workload phụ thuộc

In [ ]:
STAGE_PLAN = {"schema_version": 1, "status": "FROZEN", "decision_authority": "feature_parity_workload_only", "input_frames": EXPECTED_FEATURE_FRAMES, "groups": {"ocr_classical": {"stages": ["n5_ocr_easyocr", "n5_ocr_paddle", "n5_ocr_vietocr"], "input": "notebook13_deterministic_jpeg", "dependency": None, "environment": "separate_from_qwen25"}, "qwen25": {"stages": ["n6_caption_qwen", "n6_ocr_qwen25"], "input": "notebook13_deterministic_jpeg", "dependency": None, "environment": "vllm_isolated"}, "bge_m3": {"stages": ["n6_bge_m3"], "input": "verified_qwen25_caption", "dependency": "qwen25.COMPLETE", "environment": "text_embedding_isolated"}}, "full_parity_required_before_official_ab": True}
STAGE_PLAN["stage_plan_payload_hash"] = sha256_json(STAGE_PLAN)
PARITY_ROWS = []
for base in WORKLIST:
    runner = RUNNER_BY_KEYFRAME[base["keyframe_id"]]
    paths = SOURCE_ARTIFACTS[base["video_id"]]
    row = {**base, "runner": runner, "source_images_tar": paths["images"], "source_image_manifest": paths["image_manifest"], "source_siglip": paths["siglip"], "source_report": paths["report"], "source_marker": paths["marker"], "source_marker_payload_hash": MARKER_HASHES[base["video_id"]], "required_groups": ["ocr_classical", "qwen25", "bge_m3"]}
    row["parity_work_item_hash"] = sha256_json(row)
    PARITY_ROWS.append(row)
PARITY_ROWS.sort(key=lambda row: (row["video_id"], int(row["frame_idx"])))
atomic_write_json(OUTPUT / "stage-plan.json", STAGE_PLAN)
atomic_write_jsonl(OUTPUT / "feature-parity-worklist.jsonl", PARITY_ROWS)
for runner in RUNNERS:
    atomic_write_jsonl(OUTPUT / f"feature-parity-worklist-{runner}.jsonl", (row for row in PARITY_ROWS if row["runner"] == runner))
ledger = {"schema_version": 1, "status": "PASS", "repo_commit": REPO_COMMIT, "selection_freeze_sha256": EXPECTED_SELECTION_FREEZE_SHA256, "nb22_complete_sha256": EXPECTED_NB22_COMPLETE_SHA256, "runner_audit": RUNNER_AUDIT, "marker_payload_hashes_sha256": sha256_json(MARKER_HASHES), "feature_frames": len(PARITY_ROWS), "heldout_accessed": False, "btc_mutated": False}
ledger["input_ledger_payload_hash"] = sha256_json(ledger)
atomic_write_json(OUTPUT / "input-ledger.json", ledger)
names = ("stage-plan.json", "input-ledger.json", "feature-parity-worklist.jsonl", "feature-parity-worklist-quang.jsonl", "feature-parity-worklist-trong.jsonl")
freeze = {"schema_version": 1, "status": "READY_FOR_FEATURE_PARITY_PRODUCERS", "decision_authority": "workload_freeze_only", "repo_commit": REPO_COMMIT, "feature_frames": len(PARITY_ROWS), "source_videos": len(SOURCE_ARTIFACTS), "runners": list(RUNNERS), "selection_freeze_sha256": EXPECTED_SELECTION_FREEZE_SHA256, "nb22_complete_sha256": EXPECTED_NB22_COMPLETE_SHA256, "stage_plan_payload_hash": STAGE_PLAN["stage_plan_payload_hash"], "files": {name: sha256_file(OUTPUT / name) for name in names}, "full_feature_parity_status": "BLOCKED_PENDING_OCR_QWEN25_BGE_AND_DOWNSTREAM_VALIDATION", "official_ab_status": "BLOCKED_PENDING_FULL_FEATURE_PARITY", "heldout_accessed": False, "btc_mutated": False, "created_at_utc": datetime.now(timezone.utc).isoformat()}
freeze["freeze_payload_hash"] = sha256_json(freeze)
atomic_write_json(OUTPUT / "parity-freeze.json", freeze)
print({"parity_rows": len(PARITY_ROWS), "runner_rows": {runner: sum(row["runner"] == runner for row in PARITY_ROWS) for runner in RUNNERS}, "stage_plan": STAGE_PLAN})

## 5 — Publish research-only

In [ ]:
files = sorted(path for path in OUTPUT.iterdir() if path.is_file())
research.put_many([(path, f"{output_root}/{path.name}") for path in files])
for path in files:
    if research.sha256(f"{output_root}/{path.name}") != sha256_file(path):
        raise RuntimeError(f"Remote verify failed: {path.name}")
complete = {"schema_version": 1, "status": "COMPLETE", "output_status": "READY_FOR_FEATURE_PARITY_PRODUCERS", "repo_commit": REPO_COMMIT, "feature_frames": EXPECTED_FEATURE_FRAMES, "runners": list(RUNNERS), "parity_freeze_sha256": sha256_file(OUTPUT / "parity-freeze.json"), "files": {path.name: sha256_file(path) for path in files}, "next": "Run separate resumable OCR-classical and Qwen25 producers; run BGE-M3 only after Qwen25 COMPLETE.", "heldout_accessed": False, "btc_mutated": False, "completed_at_utc": datetime.now(timezone.utc).isoformat()}
complete["manifest_payload_hash"] = sha256_json(complete)
complete_path = OUTPUT / "COMPLETE.json"
atomic_write_json(complete_path, complete)
research.put(complete_path, f"{output_root}/COMPLETE.json")
if research.sha256(f"{output_root}/COMPLETE.json") != sha256_file(complete_path):
    raise RuntimeError("Remote COMPLETE verify failed")
print({"FEATURE_PARITY_WORKLIST_FROZEN": True, "frames": EXPECTED_FEATURE_FRAMES, "runners": len(RUNNERS), "source_videos": len(SOURCE_ARTIFACTS), "remote_output": output_root, "parity_freeze_sha256": complete["parity_freeze_sha256"], "complete_sha256": sha256_file(complete_path), "next": complete["next"]})

## 6 — Tiếp theo

OCR cổ điển và Qwen25 có thể chạy song song theo Quang/Trọng. BGE-M3 phải đợi
caption Qwen25 COMPLETE. Sau đó còn một aggregate parity/downstream fingerprint
gate; gate đó PASS mới được chạy factorized B0–B4 DEV A/B.